# Molecular Water Solubility: ESOL Model Comparison

## Research question
Which molecular representation and regression model best predict the measured aqueous solubility of small organic molecules, and how does performance change for unseen molecular scaffolds?

The response is Delaney ESOL measured log solubility in mol/L. Lower MAE and RMSE are better; $R^2$ is also reported.

## Experimental design
Compare Random Forest, Extra Trees, and Histogram Gradient Boosting using 20 fixed RDKit descriptors and radius-2 Morgan fingerprints. Each random/scaffold split selects a candidate on validation MAE. Only that candidate is scored on its held-out test set. A mean predictor is included as a baseline.

The scaffold split groups Bemis-Murcko scaffolds. Acyclic molecules with empty scaffolds receive individual canonical-SMILES groups; this fragments acyclic chemical space and should be considered when interpreting generalization.

The cells run the repository's offline checks and then the full public-data pipeline. The full run downloads ESOL if it is not already present.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
import solubility_project

solubility_project.run_self_tests()

In [ ]:
summary = solubility_project.run_pipeline(
    data_dir='data',
    output_dir='outputs/solubility',
    seed=42,
    fingerprint_bits=2048,
    n_jobs=-1,
)
summary['selections']

## Results
Validation results are used for model selection. Test metrics below correspond only to the selected candidate for each split; do not use these test scores for tuning.

In [ ]:
output_dir = Path('outputs/solubility')
validation_results = pd.read_csv(output_dir / 'validation_results.csv')
test_results = pd.read_csv(output_dir / 'test_results.csv')
display(validation_results.sort_values(['split', 'mae']))
display(test_results)

In [ ]:
display(Image(filename=str(output_dir / 'validation_mae.png')))
display(Image(filename=str(output_dir / 'test_parity.png')))

## Interpretation checklist
- Compare the scaffold test MAE with random test MAE; a larger scaffold error suggests weaker generalization to unseen chemotypes.
- Check whether descriptors or fingerprints are consistently selected, and compare their validation ranking across split strategies.
- Compare selected models against the mean baseline and discuss the parity plots, target range, and outliers.
- Report the source checksum, row-cleaning counts, seed, and saved summary from `outputs/solubility/experiment_summary.json`.
- Treat the results as a benchmark, not experimental or clinical advice.